# Validation
Compare the gold tables with the results of the original pandas + SQLite project
(`results/*.csv` in openalex-galician-universities), stored in the `reference` volume.

In [0]:
-- Reference results with the same columns as each gold table (_rescued_data left out)
CREATE OR REPLACE TEMPORARY VIEW ref_output_by_year AS
SELECT university, year, articles_reviews, conference_papers, total
FROM read_files('/Volumes/workspace/openalex/reference/01_output_by_year.csv', format => 'csv', header => true);

-- 02 and 03 together, as in gold_open_access
CREATE OR REPLACE TEMPORARY VIEW ref_open_access AS
SELECT a.university, a.year, a.n_works, a.n_oa, a.pct_oa,
       b.pct_gold, b.pct_diamond, b.pct_hybrid, b.pct_bronze, b.pct_green, b.pct_closed
FROM read_files('/Volumes/workspace/openalex/reference/02_open_access.csv', format => 'csv', header => true) AS a
JOIN read_files('/Volumes/workspace/openalex/reference/03_open_access_by_route.csv', format => 'csv', header => true) AS b
  ON a.university = b.university AND a.year = b.year;

CREATE OR REPLACE TEMPORARY VIEW ref_international_collaboration AS
SELECT university, year, n_art_rev, n_int_art_rev, pct_int_art_rev, n_conf, n_int_conf, pct_int_conf
FROM read_files('/Volumes/workspace/openalex/reference/04_international_collaboration.csv', format => 'csv', header => true);

CREATE OR REPLACE TEMPORARY VIEW ref_citation_impact AS
SELECT university, year, n_works, mean_fwci, median_fwci, pct_top10, pct_top1
FROM read_files('/Volumes/workspace/openalex/reference/05_citation_impact.csv', format => 'csv', header => true);

CREATE OR REPLACE TEMPORARY VIEW ref_galician_collaboration AS
SELECT uni_a, uni_b, year, n_shared
FROM read_files('/Volumes/workspace/openalex/reference/06_galician_collaboration.csv', format => 'csv', header => true);

In [0]:
-- For each gold table: row counts and rows that appear on only one side
CREATE OR REPLACE TEMPORARY VIEW validation_results AS
SELECT 'gold_output_by_year' AS gold_table,
       (SELECT COUNT(*) FROM workspace.openalex.gold_output_by_year) AS n_gold,
       (SELECT COUNT(*) FROM ref_output_by_year)                     AS n_reference,
       (SELECT COUNT(*) FROM (SELECT * FROM workspace.openalex.gold_output_by_year EXCEPT SELECT * FROM ref_output_by_year)) AS only_in_gold,
       (SELECT COUNT(*) FROM (SELECT * FROM ref_output_by_year EXCEPT SELECT * FROM workspace.openalex.gold_output_by_year)) AS only_in_reference
UNION ALL
SELECT 'gold_open_access',
       (SELECT COUNT(*) FROM workspace.openalex.gold_open_access),
       (SELECT COUNT(*) FROM ref_open_access),
       (SELECT COUNT(*) FROM (SELECT * FROM workspace.openalex.gold_open_access EXCEPT SELECT * FROM ref_open_access)),
       (SELECT COUNT(*) FROM (SELECT * FROM ref_open_access EXCEPT SELECT * FROM workspace.openalex.gold_open_access))
UNION ALL
SELECT 'gold_international_collaboration',
       (SELECT COUNT(*) FROM workspace.openalex.gold_international_collaboration),
       (SELECT COUNT(*) FROM ref_international_collaboration),
       (SELECT COUNT(*) FROM (SELECT * FROM workspace.openalex.gold_international_collaboration EXCEPT SELECT * FROM ref_international_collaboration)),
       (SELECT COUNT(*) FROM (SELECT * FROM ref_international_collaboration EXCEPT SELECT * FROM workspace.openalex.gold_international_collaboration))
UNION ALL
SELECT 'gold_citation_impact',
       (SELECT COUNT(*) FROM workspace.openalex.gold_citation_impact),
       (SELECT COUNT(*) FROM ref_citation_impact),
       (SELECT COUNT(*) FROM (SELECT * FROM workspace.openalex.gold_citation_impact EXCEPT SELECT * FROM ref_citation_impact)),
       (SELECT COUNT(*) FROM (SELECT * FROM ref_citation_impact EXCEPT SELECT * FROM workspace.openalex.gold_citation_impact))
UNION ALL
SELECT 'gold_galician_collaboration',
       (SELECT COUNT(*) FROM workspace.openalex.gold_galician_collaboration),
       (SELECT COUNT(*) FROM ref_galician_collaboration),
       (SELECT COUNT(*) FROM (SELECT * FROM workspace.openalex.gold_galician_collaboration EXCEPT SELECT * FROM ref_galician_collaboration)),
       (SELECT COUNT(*) FROM (SELECT * FROM ref_galician_collaboration EXCEPT SELECT * FROM workspace.openalex.gold_galician_collaboration));

SELECT * FROM validation_results;

In [0]:
-- Fail the notebook if any gold table differs from the reference results
SELECT assert_true(
  n_gold = n_reference AND only_in_gold = 0 AND only_in_reference = 0,
  'Gold table differs from the reference results: ' || gold_table
)
FROM validation_results;